# Understanding components of custom data loader in pytorch

## Recap - Creating Linear regression model

In [ ]:
## Dataset used
# https://www.kaggle.com/datasets/mirichoi0218/insurance

In [ ]:
# !pip install kaggle

In [ ]:
import os
import kagglehub
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Download latest version of dataset
path = kagglehub.dataset_download('mirichoi0218/insurance')
print("Path of dataset file:", path)

Using Colab cache for faster access to the 'insurance' dataset.
Path of dataset file: /kaggle/input/insurance


In [ ]:
file_name = os.listdir(path)
file_name

['insurance.csv']

In [ ]:
df = pd.read_csv(os.path.join(path, file_name[0]))
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
# Split train and test data frames
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Split dataset before encoding
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Encode categorical variables
label_encoders = {}
for col in ['sex','smoker','region']:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le # store encoders for later use

In [ ]:
# Split independent and dependent features
X_train = train_df.drop(columns=['charges'])
y_train = train_df['charges']
X_test = test_df.drop(columns=['charges'])
y_test = test_df['charges']

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1070, 6), (268, 6), (1070,), (268,))

In [ ]:
# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Convert numpy array to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [ ]:
X_train_tensor.shape, X_test_tensor.shape, y_train_tensor.shape, y_test_tensor.shape

(torch.Size([1070, 6]),
 torch.Size([268, 6]),
 torch.Size([1070, 1]),
 torch.Size([268, 1]))

In [ ]:
# Define a regression model class
class SimpleNNRegressionModel(nn.Module):
    def __init__(self, input_dim): # initialize the model; input_dim is the number of input features
        super(SimpleNNRegressionModel, self).__init__() # initialize the parent nn.Module class
        self.network = nn.Sequential(                   # create a sequence of layers that run one after another
                            nn.Linear(input_dim, 64),   # fully connected layer: input_dim features -> 64 neurons
                            nn.ReLU(),                  # apply ReLU activation: negative values become 0
                            nn.Linear(64, 128),         # fully connected layer: 64 neurons -> 128 neurons
                            nn.ReLU(),                  # apply ReLU activation again
                            nn.Linear(128, 1))          # output layer: 128 neurons -> 1 predicted value

    # Define the forward pass; x is the input batch
    def forward(self, x):
        return self.network(x) # pass x through all layers and return the prediction

In [ ]:
# Initialize model
input_dim = X_train_tensor.shape[1]
model = SimpleNNRegressionModel(input_dim)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(params=model.parameters(), lr=0.01)

**EPOCH, BATCH_SIZE AND ITERATION**  

Assume I have a dataset containing 1,000,000 rows.  

**1. Batch size**  
`batch_szie = 1000`  
The model processes 1,000 rows at a time, calculates the loss, and updates its weights. It then moves to the next batch.

**2. Iteration**  
An iteration (or training step) is one batch being processed and the model's weights being updated once.
$\displaystyle \boxed {\text{Iterations per epoch} = \frac{1,000,000}{1,000} = 1,000}$  

> So one epoch contains 1,000 iterations.

**3. Epoch**  
An epoch means the model has gone through the entire training dataset once.  
`epochs = 10`  


```text
1,000,000 training rows
          |
          v
  Batch 1: from rows 1 to 1,000       -> Iteration 1 -> Update weights
  Batch 2: from rows 1,001 to 2,000   -> Iteration 2 -> Update weights
  Batch 3: from rows 2,001 to 3,000   -> Iteration 3 -> Update weights
          ...
  Batch 1000                  -> Iteration 1000 -> Update weights
          |
          v
      Epoch 1 complete
```

**Conclusion**  
With 1,000,000 rows, `batch_size = 1,000` and `epochs = 10`:
1. The model processes the training data in batches by batche.
2. Each batch includes 1000 rows (with one weight update).
3. It takes 1,000 iterations to process all 1,000,000 rows once.
4. Completing that full pass is 1 epoch.
5. The model repeats this process for 10 epochs, so it sees the training dataset 10 times.

In [ ]:
# Training loop: repeatedly train the model using the training data
epochs = 1000 # number of times the model will go through the training loop

# Repeat the training process 30,000 times
for epoch in range(epochs):
    model.train() # put the model in training mode
    # NOTE: This matters for layers such as Dropout and BatchNorm

    # Clear gradients left over from the previous training step
    optimizer.zero_grad() # PyTorch accumulates gradients, so we reset them each time

    # Feed training inputs into the model
    predictions = model(X_train_tensor) # the model produces its predicted values

    # Compare predictions with the correct answers
    loss = criterion(predictions, y_train_tensor)  # the loss measures how wrong the predictions are

    # Calculate how each model parameter contributed to the error
    loss.backward() # this computes gradients using backpropagation

    # Update the model's weights using the calculated gradients
    optimizer.step()  # the optimizer adjusts the weights to try to reduce the loss

    # Run this block after every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.3f}")
        # Print the current epoch and loss
        # .item() converts the loss tensor into a regular Python number
        # :.3f displays the loss with 4 digits after the decimal point

Epoch [100/1000], Loss: 46063608.000
Epoch [200/1000], Loss: 31492034.000
Epoch [300/1000], Loss: 28757394.000
Epoch [400/1000], Loss: 26613094.000
Epoch [500/1000], Loss: 25004212.000
Epoch [600/1000], Loss: 23495168.000
Epoch [700/1000], Loss: 22003528.000
Epoch [800/1000], Loss: 21106980.000
Epoch [900/1000], Loss: 20640362.000
Epoch [1000/1000], Loss: 20317534.000


## Understanding Components of a Custom DataLoader in PyTorch

1. `torch.utils.data.Dataset` $\to$ defines how to access individual samples in your data
2. `torch.utils.data.DataLoader` $\to$ takes a Dataset and provides samples to the model in batches

**Steps we practice here**  
- `__init__()` $\to$ runs when we create an instance of the dataset
- `__len__()` $\to$ returns how many samples are in the dataset
- `__getitem__()` $\to$ returns the sample and its target at a specific index.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Define a custom PyTorch Dataset for the insurance data
class InsuranceDataset(Dataset):

    # Initialize the dataset with features (X) and targets (y)
    def __init__(self, X, y):
        self.X = X  # Store the input features
        self.y = y  # Store the target values

    # Return the total number of samples in the dataset
    def __len__(self):
        return len(self.X)

    # Retrieve one sample using its index
    def __getitem__(self, idx):

        # Get the features at the specified index
        # Convert them into a PyTorch tensor with float32 data type
        features = torch.tensor(self.X[idx], dtype=torch.float32)

        # Convert the target values into a PyTorch tensor
        target = torch.tensor(self.y.iloc[idx], dtype=torch.float32)

        # Return the features and target for this sample
        return features, target

There are three useful terms to understand:


| Term	| Meaning |
|-------|---------|
|Function | A reusable block of code defined independently or inside another function.|
| Method | A function defined inside a class.|
| Instance method | A method that receives the object itself through self.|


Here all three methods; `__init__()`, `__len()__` and `__getitem__()` are **instance methods**.  

In [ ]:
dataset = InsuranceDataset(X_train, y_train)
print(f"Number of samples: {len(dataset)}")
print(f"First sample: {dataset[0]}")
print(f"Shape of X data: {dataset.X.shape}")
print(f"Shape of y data: {dataset.y.shape}")
print(f"Get first sample: {dataset.__getitem__(0)}")

Number of samples: 1070
First sample: (tensor([ 0.4722, -1.0246, -1.7565,  0.7343, -0.5087, -0.4561]), tensor(9193.8389))
Shape of X data: (1070, 6)
Shape of y data: (1070,)
Get first sample: (tensor([ 0.4722, -1.0246, -1.7565,  0.7343, -0.5087, -0.4561]), tensor(9193.8389))


Creates a DataLoader object that retrieves samples from your `InsuranceDataset` and organizes them into batches for model training.

The DataLoader uses its methods:
- `__len__()` to determine the number of samples.
- `__getitem__()` to retrieve individual samples.

Those methods are called **Python special methods** (also called magic methods).

In [ ]:
# Create a DataLoader to load the training dataset in batches
dataloader = DataLoader(
                    dataset,          # The Dataset object containing training features and targets
                    batch_size=32,    # Number of samples to retrieve in each batch
                    shuffle=True,     # Randomly shuffle sample indices at the beginning of each epoch
                    num_workers=4     # Use 4 worker processes to load and prepare data in parallel
                    )

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


**Four worker operation**  
![picture](./data/4_worker_operation.jpg)

In [38]:
for batch_idx, (features, targets) in enumerate(dataloader, start=1):
    print(f"Batch: {batch_idx}")
    print(f"Shape of features: {features.shape}")
    print(f"Shape of targets: {targets.shape}")
    break

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Batch: 1
Shape of features: torch.Size([32, 6])
Shape of targets: torch.Size([32])


In [44]:
len(dataset)/ 32

# There are 34 times of 32 batches inside dataloader.

33.4375

In [46]:
# Set the total number of training epochs (complete passes through the dataset)
epochs = 1000

# Loop through the entire dataset 1,000 times
for epoch in range(1, epochs+1):

    # Set the model to training mode
    # This enables training-specific behavior, such as Dropout and Batch Normalization updates
    model.train()

    # Iterate through the DataLoader one batch at a time
    # enumerate(start=1) makes the batch counter begin at 1
    # batch_X contains the input features, and batch_y contains the corresponding targets
    for batch_idx, (batch_X, batch_y) in enumerate(dataloader, start=1):

        # Display the current batch number
        print(f"Current batch : {batch_idx}")

        # Clear gradients from the previous batch
        # PyTorch accumulates gradients by default, so this must be called before backpropagation
        optimizer.zero_grad()

        # Forward pass: pass the current batch of features through the model
        # The model produces predictions for the samples in this batch
        predictions = model(batch_X)

        # Calculate the loss by comparing predictions with the actual target values
        loss = criterion(predictions, batch_y)

        # Backward pass: calculate gradients of the loss with respect to model parameters
        loss.backward()

        # Update the model parameters (weights and biases) using the calculated gradients
        optimizer.step()

        # Display the current batch number and its loss
        # loss.item() converts the loss tensor into a regular Python number
        print(f'Batch [{batch_idx}/{len(dataloader)}], Loss: {loss.item():.3f}')

    # Check whether 100 epochs have been completed
    # epoch starts from 0, so epoch + 1 gives the actual epoch number
    if (epoch) % 100 == 0:

        # Display the epoch number and the loss from the final batch of this epoch
        print(f'Epoch [{epoch}/{epochs}], Loss: {loss.item():.4f}')

Streaming output truncated to the last 5000 lines.
Batch [17/34], Loss: 130791632.000
Current batch : 18
Batch [18/34], Loss: 122854488.000
Current batch : 19
Batch [19/34], Loss: 184551664.000
Current batch : 20
Batch [20/34], Loss: 122896032.000
Current batch : 21
Batch [21/34], Loss: 98643952.000
Current batch : 22
Batch [22/34], Loss: 154968864.000
Current batch : 23
Batch [23/34], Loss: 105940848.000
Current batch : 24
Batch [24/34], Loss: 179612544.000
Current batch : 25
Batch [25/34], Loss: 125446688.000
Current batch : 26
Batch [26/34], Loss: 101117992.000
Current batch : 27
Batch [27/34], Loss: 101891304.000
Current batch : 28
Batch [28/34], Loss: 106516080.000
Current batch : 29
Batch [29/34], Loss: 133367600.000
Current batch : 30
Batch [30/34], Loss: 94585848.000
Current batch : 31
Batch [31/34], Loss: 164298368.000
Current batch : 32
Batch [32/34], Loss: 134609248.000
Current batch : 33
Batch [33/34], Loss: 168315936.000
Current batch : 34
Batch [34/34], Loss: 103185952.00